# 🐍 Giornata aggiuntiva: analisi dati con pandas e grafici con matplotlib
**Durata:** 8 ore (circa 7 ore effettive) · **Prerequisiti:** giornate 1–5 (soprattutto liste, dizionari, funzioni, lambda, file, CSV, JSON)

**Come si usa questo notebook**
- Esegui le celle con `Shift + Invio`. Le celle **🧪 Esercizio** vanno completate sostituendo i `# TODO`.
- Livelli: ⭐ base, ⭐⭐ intermedio, ⭐⭐⭐ sfida. Chi finisce prima passa al livello successivo.
- I dati usati sono **inventati a scopo didattico**: non rappresentano voli, materiali o consumi reali.

**Programma della giornata**
0. Warm-up e installazione (30 min)
1. Introduzione a pandas: `Series` e `DataFrame` (1 h)
2. Leggere e scrivere dati (45 min)
3. Selezione e filtri (1 h)
4. Pulizia dei dati (30 min)
5. Aggregazioni e unione di tabelle (1 h)
6. Grafici con matplotlib (1 h 30)
7. Laboratorio: un piccolo cruscotto di analisi (45 min)
8. Riepilogo

## 0. Warm-up e installazione

### 0.1 Installare le librerie
**pandas** serve per lavorare con dati in forma di tabella; **matplotlib** serve per disegnare grafici. Si installano nell'ambiente virtuale del progetto, con il venv attivo, da terminale:
```
pip install pandas matplotlib
```
(`numpy`, la libreria numerica su cui pandas si basa, viene installata da sola). In alternativa, da una cella del notebook: `%pip install pandas matplotlib`.

In PyCharm: *Settings → Python → Interpreter → +* e cerca `pandas` e `matplotlib`. Se dopo l'installazione l'import non funziona, riavvia il kernel del notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print("pandas", pd.__version__)
import matplotlib
print("matplotlib", matplotlib.__version__)

Per convenzione, si importa sempre con questi nomi abbreviati: `pd` e `plt`.

### 0.2 Warm-up: cosa ricordo?
**Prima di eseguire** ogni cella, scrivi cosa ti aspetti che stampi.

In [ ]:
# 1
import csv, io
testo = "codice,passeggeri\nAZ101,120\nLH202,85\n"
righe = list(csv.DictReader(io.StringIO(testo)))
print(righe[0]["codice"], int(righe[1]["passeggeri"]) + 1)

In [ ]:
# 2
import json
print(json.dumps({"a": 1, "b": [True, None]}))

In [ ]:
# 3
class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def somma(self):
        return self.x + self.y

print(Punto(3, 4).somma())

In [ ]:
# 4
voli = [("AZ", 120), ("LH", 85), ("AF", 150)]
print(max(voli, key=lambda v: v[1]))

In [ ]:
# 5
from datetime import date
print((date(2026, 11, 19) - date(2026, 11, 1)).days)

*Risposte:* 1) `AZ101 86` · 2) `{"a": 1, "b": [true, null]}` · 3) `7` · 4) `('AF', 150)` · 5) `18`

## 1. Introduzione a pandas

**pandas** gestisce dati in forma di **tabella**: righe e colonne, con etichette. È lo strumento standard in Python per ripulire, esplorare e riassumere dati (file CSV, fogli di calcolo, risultati di sondaggi, misure di laboratorio).

Due strutture:
- **`Series`**: una singola colonna (una lista con etichette e un tipo uniforme).
- **`DataFrame`**: una tabella, cioè un insieme di colonne `Series` che condividono lo stesso indice (le etichette delle righe).

### 1.1 `Series`

In [ ]:
voti = pd.Series([27, 30, 24, 28], index=["Anna", "Luca", "Sara", "Marco"])
print(voti)
print(voti["Luca"])          # accesso per etichetta
print(voti.iloc[0])          # accesso per posizione
print(voti.mean(), voti.max(), voti.sum())

La caratteristica più importante: le operazioni sono **vettoriali**, cioè si applicano a tutti gli elementi in una volta, senza cicli `for`.

In [ ]:
print(voti + 1)              # aggiunge 1 a tutti
print(voti > 26)             # serie di True/False
print(voti[voti > 26])       # filtra: tiene solo i voti > 26

### 1.2 `DataFrame`
Si costruisce da un dizionario (una chiave per colonna) o da una lista di dizionari (uno per riga).

In [ ]:
studenti = pd.DataFrame({
    "nome": ["Anna", "Luca", "Sara", "Marco"],
    "voto": [27, 30, 24, 28],
    "corso": ["fisica", "chimica", "fisica", "informatica"],
})
print(studenti)

In [ ]:
da_righe = pd.DataFrame([
    {"nome": "Anna", "voto": 27},
    {"nome": "Luca", "voto": 30},
])
print(da_righe)

### 1.3 Esplorare una tabella
| Comando | Cosa mostra |
|---|---|
| `df.head(n)` / `df.tail(n)` | prime / ultime `n` righe (5 se omesso) |
| `df.shape` | `(righe, colonne)` |
| `df.columns` | nomi delle colonne |
| `df.dtypes` | tipo di ogni colonna |
| `df.info()` | riepilogo: tipi, valori non nulli, memoria |
| `df.describe()` | statistiche delle colonne numeriche |

*Nota:* i tipi di testo possono comparire come `object` o come `str` a seconda della versione di pandas: sono equivalenti per i nostri scopi.

In [ ]:
print(studenti.shape)
print(list(studenti.columns))
print(studenti.dtypes)
studenti.info()
print(studenti.describe())

### 1.4 Colonne: leggere, creare, modificare
- `df["col"]` restituisce una colonna (una `Series`); `df[["a", "b"]]` (doppie parentesi) restituisce una **tabella** con due colonne.
- Assegnando a una colonna nuova si **crea** la colonna; l'operazione si applica a tutte le righe.

In [ ]:
print(studenti["voto"])
print(studenti[["nome", "voto"]])

studenti["voto_su_100"] = studenti["voto"] / 30 * 100
studenti["promosso"] = studenti["voto"] >= 18
print(studenti)

#### 🧪 Esercizio – Una serie di temperature  ⭐
Crea una `Series` con le temperature `[18.5, 21.0, 19.5, 24.0]` e indice `["lun", "mar", "mer", "gio"]`. Stampa la media e le sole temperature superiori alla media.

In [ ]:
import pandas as pd
# TODO

#### 🧪 Esercizio – Tabella di materiali  ⭐
Crea un `DataFrame` con colonne `materiale` (`["acciaio", "alluminio", "titanio"]`) e `densita` (`[7850, 2700, 4506]`). Stampa forma, nomi delle colonne e `describe()`.

In [ ]:
import pandas as pd
# TODO

#### 🧪 Esercizio – Colonna calcolata  ⭐
Nel `DataFrame` precedente aggiungi la colonna `densita_g_cm3` (densità divisa per 1000) e stampa la tabella.

In [ ]:
# TODO

#### 🧪 Esercizio – Più colonne  ⭐
Dato `studenti`, stampa la tabella con le sole colonne `nome` e `corso`. Poi stampa solo la colonna `corso` e spiega nel commento la differenza tra i due risultati.

In [ ]:
# TODO

## 2. Leggere e scrivere dati

### 2.1 I dati del corso
Questa cella crea quattro file CSV con dati d'esempio (voli, materiali, consumi energetici di una casa, aeroporti). **Eseguila una sola volta.** I dati sono inventati.

In [ ]:
voli_dati = [
    ("AZ101", "ITA", "TRN", "FCO", 650, 120, 5, "2026-10-01"),
    ("AZ102", "ITA", "FCO", "TRN", 650, 98, 0, "2026-10-01"),
    ("LH202", "Lufthansa", "TRN", "FRA", 1200, 85, 15, "2026-10-02"),
    ("AF303", "AirFrance", "TRN", "CDG", 480, 150, 0, "2026-10-02"),
    ("BA404", "BritishAirways", "TRN", "LHR", 1800, 60, 45, "2026-10-03"),
    ("IB505", "Iberia", "TRN", "MAD", 950, 110, 10, "2026-10-03"),
    ("AZ103", "ITA", "TRN", "NAP", 880, 130, 25, "2026-10-04"),
    ("LH203", "Lufthansa", "FRA", "TRN", 1200, 92, 0, "2026-10-04"),
    ("AF304", "AirFrance", "CDG", "TRN", 480, 140, 5, "2026-10-05"),
    ("BA405", "BritishAirways", "LHR", "TRN", 1800, 75, 30, "2026-10-05"),
    ("IB506", "Iberia", "MAD", "TRN", 950, 100, 0, "2026-10-06"),
    ("AZ104", "ITA", "NAP", "TRN", 880, 125, 10, "2026-10-06"),
    ("FR606", "Ryanair", "TRN", "BCN", 780, 180, 20, "2026-10-07"),
    ("FR607", "Ryanair", "BCN", "TRN", 780, 175, 0, "2026-10-07"),
    ("LH204", "Lufthansa", "TRN", "MUC", 560, 70, 35, "2026-10-08"),
    ("LH205", "Lufthansa", "MUC", "TRN", 560, 68, 5, "2026-10-08"),
    ("AZ105", "ITA", "TRN", "FCO", 650, 110, 0, "2026-10-09"),
    ("AF305", "AirFrance", "TRN", "CDG", 480, 145, 12, "2026-10-09"),
    ("BA406", "BritishAirways", "TRN", "LHR", 1800, 80, 60, "2026-10-10"),
    ("IB507", "Iberia", "TRN", "MAD", 950, 105, 8, "2026-10-10"),
]
pd.DataFrame(voli_dati, columns=["codice", "compagnia", "partenza", "arrivo", "distanza_km", "passeggeri", "ritardo_min", "giorno"]).to_csv("voli.csv", index=False)

materiali_dati = [
    ("acciaio", "metallo", 7850, 400, 1.2),
    ("alluminio", "metallo", 2700, 310, 2.8),
    ("titanio", "metallo", 4506, 900, 35.5),
    ("rame", "metallo", 8960, 220, 8.5),
    ("nylon", "polimero", 1150, 80, 3.5),
    ("polietilene", "polimero", 950, 25, 1.4),
    ("pvc", "polimero", 1400, 50, 1.1),
    ("allumina", "ceramica", 3950, 300, 15.0),
    ("vetro", "ceramica", 2500, 50, 1.0),
    ("fibra_carbonio", "composito", 1600, 1500, 45.0),
]
pd.DataFrame(materiali_dati, columns=["materiale", "famiglia", "densita", "resistenza_mpa", "costo_eur_kg"]).to_csv("materiali.csv", index=False)

consumi_dati = {
    "mese": list(range(1, 13)),
    "consumo_kwh": [520, 480, 410, 350, 300, 290, 330, 340, 310, 380, 450, 530],
    "temp_media": [4.2, 6.1, 10.3, 14.0, 18.5, 23.1, 26.0, 25.4, 20.2, 14.6, 8.9, 4.8],
}
pd.DataFrame(consumi_dati).to_csv("consumi.csv", index=False)

aeroporti_dati = [
    ("TRN", "Torino", "Italia"), ("FCO", "Roma", "Italia"), ("NAP", "Napoli", "Italia"),
    ("FRA", "Francoforte", "Germania"), ("MUC", "Monaco", "Germania"),
    ("CDG", "Parigi", "Francia"), ("LHR", "Londra", "Regno Unito"),
    ("MAD", "Madrid", "Spagna"), ("BCN", "Barcellona", "Spagna"),
]
pd.DataFrame(aeroporti_dati, columns=["iata", "citta", "paese"]).to_csv("aeroporti.csv", index=False)

### 2.2 Leggere un CSV: `pd.read_csv`
Una riga sola sostituisce il codice con `csv.DictReader` e le conversioni a mano: pandas riconosce da solo i tipi dei numeri.

In [ ]:
voli = pd.read_csv("voli.csv")
print(voli.head())
print(voli.shape)
voli.info()

Notare che la colonna `giorno` risulta testo. Con `parse_dates` pandas la riconosce come **data**.

In [ ]:
voli = pd.read_csv("voli.csv", parse_dates=["giorno"])
print(voli.dtypes)
print(voli["giorno"].dt.day_name().head(3))     # .dt dà accesso a giorno, mese, anno...
print(voli["giorno"].dt.day.head(3))

**Parametri utili di `read_csv`:**
| Parametro | Quando serve | Esempio |
|---|---|---|
| `sep` | delimitatore diverso dalla virgola | `sep=";"` |
| `decimal` | numeri con la virgola decimale | `decimal=","` |
| `encoding` | caratteri accentati illeggibili | `encoding="utf-8"` |
| `parse_dates` | colonne di date | `parse_dates=["giorno"]` |
| `usecols` | leggere solo alcune colonne | `usecols=["codice", "passeggeri"]` |
| `nrows` | leggere solo le prime righe | `nrows=100` |

I CSV esportati da Excel in italiano usano spesso `;` come separatore e `,` come decimale:

In [ ]:
testo_excel = "nome;peso_kg\nbullone;0,012\nflangia;0,340\n"
pezzi = pd.read_csv(io.StringIO(testo_excel), sep=";", decimal=",")
print(pezzi)
print(pezzi.dtypes)

### 2.3 Scrivere: `to_csv` e `to_json`
`index=False` evita di salvare l'indice come colonna in più.

In [ ]:
materiali = pd.read_csv("materiali.csv")
metalli = materiali[materiali["famiglia"] == "metallo"]

metalli.to_csv("metalli.csv", index=False)
metalli.to_json("metalli.json", orient="records", indent=2, force_ascii=False)

print(open("metalli.json", encoding="utf-8").read()[:150])
print(pd.read_json("metalli.json"))

#### 🧪 Esercizio – Leggi e osserva  ⭐
Leggi `materiali.csv` in un `DataFrame` e stampa le prime 3 righe, la forma e i tipi delle colonne.

In [ ]:
import pandas as pd
# TODO

#### 🧪 Esercizio – Solo alcune colonne  ⭐
Leggi `voli.csv` caricando **solo** le colonne `codice` e `passeggeri` (parametro `usecols`) e stampa le prime 5 righe.

In [ ]:
# TODO

#### 🧪 Esercizio – CSV all'italiana  ⭐⭐
Leggi correttamente il testo `testo = "pezzo;costo\nvite;0,05\ndado;0,08\n"` (separatore `;`, decimale `,`) usando `io.StringIO`, e stampa il costo totale.

In [ ]:
import io
testo = "pezzo;costo\nvite;0,05\ndado;0,08\n"
# TODO

#### 🧪 Esercizio – Salva un sottoinsieme  ⭐⭐
Dai `materiali`, tieni solo la famiglia `"polimero"` e salvali in `polimeri.csv` senza indice. Poi rileggi il file e stampalo.

In [ ]:
# TODO

#### 🧪 Esercizio – Da CSV a JSON  ⭐⭐
Converti `consumi.csv` in `consumi.json` con `orient="records"` e `indent=2`, poi rileggi il JSON e stampa le prime 2 righe.

In [ ]:
# TODO

## 3. Selezione e filtri

Ricarichiamo i dati principali.

In [ ]:
voli = pd.read_csv("voli.csv", parse_dates=["giorno"])
materiali = pd.read_csv("materiali.csv")
consumi = pd.read_csv("consumi.csv")
aeroporti = pd.read_csv("aeroporti.csv")
print(voli.head(3))

### 3.1 Selezionare per posizione e per etichetta
- `iloc[righe, colonne]` usa le **posizioni** (numeri, come gli slicing delle liste: l'ultimo è escluso).
- `loc[righe, colonne]` usa le **etichette** (nomi delle colonne; per le righe, l'etichetta dell'indice o una condizione). Con `loc` l'ultimo estremo è **incluso**.

In [ ]:
print(voli.iloc[0])                    # prima riga
print(voli.iloc[0:3, 0:3])             # prime 3 righe, prime 3 colonne
print(voli.loc[0:2, ["codice", "passeggeri"]])      # righe con etichetta 0, 1, 2 (incluso 2)

### 3.2 Filtrare con una condizione
Una condizione su una colonna produce una serie di `True`/`False` (una **maschera**); passandola tra parentesi quadre, si tengono le righe `True`.

In [ ]:
print(voli[voli["passeggeri"] > 140])
print(voli[voli["compagnia"] == "ITA"])

**Più condizioni:** si usano `&` (e), `|` (o), `~` (non), **ciascuna condizione tra parentesi**. ⚠️ Non si possono usare `and`, `or`, `not` e si dimenticano spesso le parentesi: darebbe un errore.

In [ ]:
print(voli[(voli["compagnia"] == "ITA") & (voli["ritardo_min"] > 0)])
print(voli[(voli["distanza_km"] > 1500) | (voli["passeggeri"] > 170)])
print(voli[~(voli["partenza"] == "TRN")].head(3))

**Altri filtri comodi**

In [ ]:
print(voli[voli["compagnia"].isin(["Iberia", "Ryanair"])])             # appartiene a un elenco
print(voli[voli["distanza_km"].between(500, 900)].head(3))             # intervallo (estremi inclusi)
print(voli[voli["codice"].str.startswith("AZ")])                       # metodi sulle stringhe

**`loc` con condizione e colonne:** seleziona righe e colonne insieme.

In [ ]:
print(voli.loc[voli["ritardo_min"] > 30, ["codice", "compagnia", "ritardo_min"]])

### 3.3 Ordinare
`sort_values(colonna, ascending=...)`; con una lista di colonne si ordina per più criteri. `nlargest` e `nsmallest` danno i primi/ultimi `n`.

In [ ]:
print(voli.sort_values("passeggeri", ascending=False).head(3))
print(voli.sort_values(["compagnia", "passeggeri"], ascending=[True, False]).head(4))
print(voli.nlargest(3, "ritardo_min")[["codice", "ritardo_min"]])

### 3.4 Creare colonne nuove
Con operazioni vettoriali, con `numpy.where` (un `if/else` su tutta la colonna) o con `apply` e una **lambda** (come nella giornata 4).

In [ ]:
import numpy as np

voli["durata_h"] = (voli["distanza_km"] / 800).round(2)
voli["in_ritardo"] = voli["ritardo_min"] > 15
voli["fascia"] = np.where(voli["distanza_km"] > 1000, "lungo", "breve")
voli["rotta"] = voli["partenza"] + "-" + voli["arrivo"]
voli["ritardo_h"] = voli["ritardo_min"].apply(lambda m: round(m / 60, 2))
print(voli[["codice", "durata_h", "in_ritardo", "fascia", "rotta", "ritardo_h"]].head())

### 3.5 Valori distinti e conteggi

In [ ]:
print(voli["compagnia"].unique())
print(voli["compagnia"].nunique())
print(voli["compagnia"].value_counts())

#### 🧪 Esercizio – Voli affollati  ⭐
Mostra codice e passeggeri dei voli con più di 130 passeggeri.

In [ ]:
# TODO

#### 🧪 Esercizio – Materiali leggeri  ⭐
Mostra nome e densità dei materiali con densità inferiore a 2000 kg/m³.

In [ ]:
# TODO

#### 🧪 Esercizio – Due condizioni  ⭐⭐
Mostra i voli della compagnia `Lufthansa` con ritardo maggiore di 10 minuti.

In [ ]:
# TODO

#### 🧪 Esercizio – Metalli resistenti  ⭐⭐
Mostra i materiali della famiglia `metallo` con resistenza superiore a 300 MPa, ordinati dal più resistente al meno resistente.

In [ ]:
# TODO

#### 🧪 Esercizio – Elenco di compagnie  ⭐⭐
Mostra i voli di `ITA` e `AirFrance` in un unico filtro (usa `isin`) e stampa quanti sono.

In [ ]:
# TODO

#### 🧪 Esercizio – Resistenza specifica  ⭐⭐
Aggiungi ai `materiali` la colonna `resistenza_specifica` = `resistenza_mpa / densita * 1000` (arrotondata a 1 decimale). Stampa i 3 materiali con il valore più alto (`nlargest`).

In [ ]:
# TODO

#### 🧪 Esercizio – Ritardo grave  ⭐⭐⭐
Crea nei `voli` la colonna `gravita`: `"nessuno"` se il ritardo è 0, `"lieve"` se è tra 1 e 15 inclusi, `"grave"` se superiore a 15. Suggerimento: `apply` con una funzione che usa `if/elif/else`. Stampa quanti voli ci sono per ciascuna gravità.

In [ ]:
# TODO

#### 🧪 Esercizio – Domanda aperta: voli della settimana  ⭐⭐⭐
Quanti voli sono partiti da `TRN` nei primi 5 giorni (dal 1 al 5 ottobre 2026 inclusi) con più di 100 passeggeri? Suggerimento: confronta `voli["giorno"]` con `pd.Timestamp("2026-10-05")` e combina tre condizioni.

In [ ]:
# TODO

## 4. Pulizia dei dati

I dati reali sono quasi sempre "sporchi": valori mancanti, spazi in più, maiuscole incoerenti, numeri salvati come testo, righe duplicate. Pulirli è gran parte del lavoro. Esempio con una piccola tabella:

In [ ]:
grezzo = pd.DataFrame({
    "nome": [" Anna", "luca ", "SARA", "Anna", None, "Marco"],
    "eta": ["25", "31", None, "25", "40", "29"],
    "citta": ["Torino", "torino", "Milano", "Torino", "Roma", None],
    "punteggio": [8.5, None, 7.0, 8.5, 6.5, 9.0],
})
print(grezzo)
print(grezzo.isna().sum())          # quanti valori mancanti per colonna

**Operazioni di pulizia principali**
| Problema | Rimedio |
|---|---|
| spazi e maiuscole incoerenti | `.str.strip()`, `.str.title()`, `.str.lower()` |
| numeri salvati come testo | `pd.to_numeric(colonna)` o `.astype(int)` |
| date come testo | `pd.to_datetime(colonna)` |
| valori mancanti | `.dropna()` (elimina righe) o `.fillna(valore)` (riempie) |
| righe duplicate | `.drop_duplicates()` |
| nomi di colonna scomodi | `.rename(columns={"vecchio": "nuovo"})` |

⚠️ **Regola di sicurezza:** i metodi di pandas non modificano la tabella: restituiscono una tabella nuova che va **riassegnata** (`df = df.dropna()`). Evita `inplace=True` e le assegnazioni "a catena" (`df[...][...] = ...`): non funzionano in modo affidabile, soprattutto nelle versioni recenti di pandas.

In [ ]:
pulito = grezzo.copy()

pulito["nome"] = pulito["nome"].str.strip().str.title()
pulito["citta"] = pulito["citta"].str.strip().str.title()
pulito["eta"] = pd.to_numeric(pulito["eta"])

pulito = pulito.dropna(subset=["nome"])                         # senza nome la riga è inutilizzabile
pulito["eta"] = pulito["eta"].fillna(pulito["eta"].median())    # età mancante: mediana
pulito["punteggio"] = pulito["punteggio"].fillna(pulito["punteggio"].mean())
pulito["citta"] = pulito["citta"].fillna("Sconosciuta")

pulito = pulito.drop_duplicates()
print(pulito)
print(pulito.isna().sum().sum(), "valori mancanti")

#### 🧪 Esercizio – Conta i mancanti  ⭐
Crea `ordini = pd.DataFrame({"prodotto": ["vite", "dado", None, "vite"], "quantita": ["10", "5", "7", None], "prezzo": [0.05, None, 0.2, 0.05]})` e stampa il numero di valori mancanti per colonna.

In [ ]:
import pandas as pd
# TODO

#### 🧪 Esercizio – Pulisci gli ordini  ⭐⭐
Sulla tabella `ordini`: converti `quantita` in numero, elimina le righe senza `prodotto`, riempi la `quantita` mancante con 0 e il `prezzo` mancante con la media dei prezzi. Stampa il risultato.

In [ ]:
# TODO

#### 🧪 Esercizio – Spazi e maiuscole  ⭐⭐
Data `colori = pd.Series(["  Rosso", "BLU ", "verde", " Rosso  "])`, normalizza i valori (senza spazi, tutto minuscolo) e stampa quanti valori diversi ci sono (`nunique`).

In [ ]:
colori = pd.Series(["  Rosso", "BLU ", "verde", " Rosso  "])
# TODO

#### 🧪 Esercizio – Date come testo  ⭐⭐
Data la colonna `date_testo = pd.Series(["21/10/2026", "26/10/2026", "28/10/2026"])`, convertila in date con `pd.to_datetime(..., format="%d/%m/%Y")` e stampa i giorni della settimana (`.dt.day_name()`).

In [ ]:
date_testo = pd.Series(["21/10/2026", "26/10/2026", "28/10/2026"])
# TODO

## 5. Aggregazioni e unione di tabelle

### 5.1 Riassunti semplici

In [ ]:
print(voli["passeggeri"].sum(), voli["passeggeri"].mean(), voli["ritardo_min"].max())
print(voli[["passeggeri", "ritardo_min"]].mean())

### 5.2 `groupby`: dividere, calcolare, ricombinare
`groupby("colonna")` divide la tabella in gruppi (uno per valore della colonna), su ciascun gruppo si calcola un riassunto, e i risultati vengono messi in una nuova tabella.

In [ ]:
print(voli.groupby("compagnia")["passeggeri"].sum())
print(voli.groupby("compagnia")["ritardo_min"].mean().round(1))

Per più calcoli insieme si usa `agg` con la **sintassi con nomi**: `nome_colonna_risultato=("colonna_origine", "funzione")`. `reset_index()` riporta il nome del gruppo a normale colonna.

In [ ]:
riepilogo = voli.groupby("compagnia").agg(
    n_voli=("codice", "count"),
    pax_totali=("passeggeri", "sum"),
    ritardo_medio=("ritardo_min", "mean"),
    ritardo_max=("ritardo_min", "max"),
).reset_index()

riepilogo["ritardo_medio"] = riepilogo["ritardo_medio"].round(1)
print(riepilogo.sort_values("pax_totali", ascending=False))

**Raggruppare per due colonne** e **tabelle pivot** (righe × colonne):

In [ ]:
print(voli.groupby(["compagnia", "fascia"])["passeggeri"].sum())

pivot = voli.pivot_table(index="compagnia", columns="fascia", values="passeggeri", aggfunc="sum", fill_value=0)
print(pivot)

### 5.3 Unire due tabelle: `merge`
Quando le informazioni sono in tabelle diverse, `merge` le unisce confrontando una colonna comune (come un *join* nei database). Qui aggiungiamo città e paese dell'aeroporto di arrivo.

`how="left"` mantiene tutte le righe della tabella di sinistra (anche senza corrispondenza).

In [ ]:
completa = voli.merge(aeroporti, left_on="arrivo", right_on="iata", how="left")
print(completa[["codice", "arrivo", "citta", "paese"]].head())
print(completa.groupby("paese")["passeggeri"].sum().sort_values(ascending=False))

`pd.concat([df1, df2])` incolla invece due tabelle con le stesse colonne una sotto l'altra (utile per unire più file CSV dello stesso tipo).

#### 🧪 Esercizio – Totali per famiglia  ⭐
Calcola per ogni famiglia di materiali la densità media, arrotondata a 0 decimali (`groupby`).

In [ ]:
# TODO

#### 🧪 Esercizio – Quanti voli per compagnia  ⭐
Stampa il numero di voli per compagnia in ordine decrescente (`value_counts`).

In [ ]:
# TODO

#### 🧪 Esercizio – Riepilogo materiali  ⭐⭐
Con `agg` e sintassi con nomi, calcola per famiglia: `n_materiali`, `costo_medio` e `resistenza_max`. Riporta il risultato a tabella con `reset_index()`.

In [ ]:
# TODO

#### 🧪 Esercizio – Passeggeri al giorno  ⭐⭐
Calcola i passeggeri totali per ogni giorno (`groupby("giorno")`) e stampa il giorno con più passeggeri (`idxmax`).

In [ ]:
# TODO

#### 🧪 Esercizio – Tabella pivot  ⭐⭐
Crea la tabella pivot dei **ritardi medi** con compagnie sulle righe e `fascia` sulle colonne (`aggfunc="mean"`, `fill_value=0`), arrotondata a 1 decimale.

In [ ]:
# TODO

#### 🧪 Esercizio – Unisci e raggruppa  ⭐⭐⭐
Unisci i voli agli aeroporti usando la colonna `partenza` (non `arrivo`) per ottenere la città di partenza, poi conta i voli per città di partenza.

In [ ]:
# TODO

#### 🧪 Esercizio – Ritardi per paese  ⭐⭐⭐
Calcola il ritardo medio dei voli in base al **paese di arrivo** (unendo con `aeroporti`), escludendo l'Italia, in ordine decrescente.

In [ ]:
# TODO

## 6. Grafici con matplotlib

**matplotlib** disegna grafici. Si usa il modulo `pyplot`, importato come `plt`. Un grafico è formato da:
- una **figura** (`fig`): la finestra/immagine intera;
- uno o più **assi** (`ax`): le aree di disegno con le scale, le etichette, i titoli.

Il modo consigliato: `fig, ax = plt.subplots()`, poi si disegna sugli assi con `ax.plot(...)`, `ax.bar(...)` ecc., si aggiungono titolo ed etichette, e si finisce con `plt.show()`.

⚠️ **Sempre**: titolo, nome degli assi (con unità di misura) e, se ci sono più serie, legenda. Un grafico senza etichette non si capisce.

### 6.1 Grafico a linee
Adatto a valori che evolvono nel tempo o in funzione di una variabile continua.

In [ ]:
fig, ax = plt.subplots()
ax.plot(consumi["mese"], consumi["consumo_kwh"], marker="o", color="tab:blue", label="Consumo")
ax.set_title("Consumo elettrico mensile")
ax.set_xlabel("Mese")
ax.set_ylabel("Consumo (kWh)")
ax.set_xticks(range(1, 13))
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

**Due serie nello stesso grafico:** si chiama `ax.plot` due volte, ognuna con la sua `label`.

In [ ]:
fig, ax = plt.subplots()
ax.plot(consumi["mese"], consumi["consumo_kwh"] / 10, marker="o", label="Consumo (kWh / 10)")
ax.plot(consumi["mese"], consumi["temp_media"], marker="s", linestyle="--", label="Temperatura (°C)")
ax.set_title("Consumo e temperatura nel corso dell'anno")
ax.set_xlabel("Mese")
ax.set_ylabel("Valore")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

### 6.2 Grafico a barre
Adatto a confrontare **categorie**. `ax.bar` per le barre verticali, `ax.barh` per quelle orizzontali (utile con etichette lunghe). `ax.bar_label` scrive i valori sopra le barre.

In [ ]:
pax = voli.groupby("compagnia")["passeggeri"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
barre = ax.bar(pax.index, pax.values, color="tab:green")
ax.bar_label(barre)
ax.set_title("Passeggeri totali per compagnia")
ax.set_xlabel("Compagnia")
ax.set_ylabel("Passeggeri")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.barh(pax.index, pax.values, color="tab:orange")
ax.invert_yaxis()                       # la più grande in alto
ax.set_title("Passeggeri totali per compagnia")
ax.set_xlabel("Passeggeri")
plt.show()

### 6.3 Grafico a dispersione (scatter)
Mostra la relazione tra **due variabili numeriche**: ogni punto è una riga. Per colorare per categoria si disegna un gruppo alla volta.

In [ ]:
colori = {"metallo": "tab:blue", "polimero": "tab:red", "ceramica": "tab:green", "composito": "tab:purple"}

fig, ax = plt.subplots(figsize=(7, 5))
for famiglia, gruppo in materiali.groupby("famiglia"):
    ax.scatter(gruppo["densita"], gruppo["resistenza_mpa"], label=famiglia, color=colori[famiglia], s=60)

for _, riga in materiali.iterrows():                   # etichetta ogni punto con il nome
    ax.annotate(riga["materiale"], (riga["densita"], riga["resistenza_mpa"]), fontsize=8, xytext=(4, 4), textcoords="offset points")

ax.set_title("Resistenza in funzione della densità")
ax.set_xlabel("Densità (kg/m³)")
ax.set_ylabel("Resistenza (MPa)")
ax.legend(title="Famiglia")
ax.grid(True, alpha=0.3)
plt.show()

### 6.4 Istogramma
Mostra **come si distribuiscono** i valori di una variabile numerica: i valori sono divisi in intervalli (*bin*) e si conta quanti ne cadono in ciascuno.

In [ ]:
fig, ax = plt.subplots()
ax.hist(voli["ritardo_min"], bins=6, color="tab:gray", edgecolor="white")
ax.set_title("Distribuzione dei ritardi")
ax.set_xlabel("Ritardo (minuti)")
ax.set_ylabel("Numero di voli")
plt.show()

### 6.5 Più grafici in una figura: `subplots(righe, colonne)`
`plt.subplots(2, 2)` crea una griglia: `ax` diventa una matrice di assi (`ax[0, 0]`, `ax[0, 1]`...). `plt.tight_layout()` evita che le scritte si sovrappongano.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 7))

ax[0, 0].plot(consumi["mese"], consumi["consumo_kwh"], marker="o")
ax[0, 0].set_title("Consumo mensile")
ax[0, 0].set_ylabel("kWh")

ax[0, 1].bar(pax.index, pax.values, color="tab:green")
ax[0, 1].set_title("Passeggeri per compagnia")
ax[0, 1].tick_params(axis="x", rotation=45)

ax[1, 0].scatter(materiali["densita"], materiali["resistenza_mpa"])
ax[1, 0].set_title("Resistenza e densità")
ax[1, 0].set_xlabel("Densità (kg/m³)")

ax[1, 1].hist(voli["ritardo_min"], bins=6, color="tab:gray", edgecolor="white")
ax[1, 1].set_title("Ritardi")
ax[1, 1].set_xlabel("minuti")

fig.suptitle("Quattro grafici in una figura")
plt.tight_layout()
plt.show()

### 6.6 Scorciatoia: disegnare direttamente da pandas
Le tabelle e le serie hanno un metodo `.plot` che usa matplotlib, utile per grafici veloci di esplorazione.

In [ ]:
pax.plot(kind="bar", title="Passeggeri per compagnia", ylabel="Passeggeri")
plt.tight_layout()
plt.show()

voli["ritardo_min"].plot(kind="hist", bins=6, title="Ritardi")
plt.show()

consumi.plot(x="mese", y="consumo_kwh", marker="o", title="Consumo mensile")
plt.show()

### 6.7 Salvare un grafico
`fig.savefig("nome.png", dpi=150)` salva l'immagine (**prima** di `plt.show()`, altrimenti il file risulta vuoto). I formati più usati: `.png` per documenti e siti, `.pdf` e `.svg` per la stampa.

In [ ]:
from pathlib import Path

Path("grafici").mkdir(exist_ok=True)
fig, ax = plt.subplots()
ax.bar(pax.index, pax.values)
ax.set_title("Passeggeri per compagnia")
fig.savefig(Path("grafici") / "passeggeri.png", dpi=150, bbox_inches="tight")
plt.show()
print(sorted(p.name for p in Path("grafici").iterdir()))

**Quale grafico scegliere?**
| Voglio mostrare... | Grafico |
|---|---|
| un andamento nel tempo | linee |
| un confronto tra categorie | barre |
| il legame tra due numeri | scatter |
| come si distribuiscono i valori | istogramma |
| la composizione di un totale | barre impilate (i grafici a torta sono sconsigliati se le categorie sono più di 4-5) |

#### 🧪 Esercizio – Linea della temperatura  ⭐
Disegna la temperatura media mensile (`consumi`) con linea e marcatori, titolo, etichette degli assi con unità e griglia.

In [ ]:
fig, ax = plt.subplots()
# TODO
plt.show()

#### 🧪 Esercizio – Barre dei voli  ⭐
Disegna il numero di voli per compagnia (`value_counts`) con barre verticali, titolo ed etichette degli assi.

In [ ]:
conteggio = voli["compagnia"].value_counts()
fig, ax = plt.subplots()
# TODO
plt.show()

#### 🧪 Esercizio – Costo dei materiali  ⭐⭐
Disegna un grafico a barre **orizzontali** del costo al kg (`costo_eur_kg`) dei materiali, ordinati dal più caro al più economico, con il più caro in alto.

In [ ]:
# TODO

#### 🧪 Esercizio – Distanza e passeggeri  ⭐⭐
Disegna uno scatter con la distanza sull'asse x e i passeggeri sull'asse y. C'è una relazione tra le due variabili? Scrivila nel commento.

In [ ]:
# TODO

#### 🧪 Esercizio – Istogramma dei passeggeri  ⭐⭐
Disegna l'istogramma dei passeggeri con 5 intervalli, bordi bianchi, titolo ed etichette.

In [ ]:
# TODO

#### 🧪 Esercizio – Due assi y  ⭐⭐⭐
Disegna nello stesso grafico il consumo (`consumo_kwh`) e la temperatura (`temp_media`) usando **due assi y** diversi: `ax2 = ax.twinx()`. Colora le due linee in modo diverso e aggiungi le etichette dei due assi.

In [ ]:
fig, ax = plt.subplots()
# TODO
plt.show()

#### 🧪 Esercizio – Griglia di grafici  ⭐⭐⭐
Crea una figura con **due grafici affiancati** (`plt.subplots(1, 2, figsize=(10, 4))`): a sinistra le barre della densità dei materiali, a destra lo scatter di costo e resistenza. Titoli ed etichette per entrambi. Salvala in `grafici/materiali.png`.

In [ ]:
from pathlib import Path
Path("grafici").mkdir(exist_ok=True)
# TODO

## 7. 🧰 Laboratorio: un piccolo cruscotto di analisi
Metti insieme pulizia, aggregazioni e grafici. Lavora in coppia: uno scrive, l'altro guida, scambiate ogni 10 minuti. Ogni passo usa le variabili del precedente.

**Obiettivo:** analizzare i voli di ottobre e produrre un riepilogo in CSV e un'immagine con quattro grafici.

#### 🧪 Esercizio – Passo 1: carica i dati  ⭐⭐
Leggi `voli.csv` convertendo `giorno` in data. Aggiungi le colonne `rotta` (partenza-arrivo) e `in_ritardo` (ritardo > 15 minuti). Stampa forma e prime 3 righe.

In [ ]:
# TODO

#### 🧪 Esercizio – Passo 2: indicatori principali  ⭐⭐
Calcola e stampa con f-string: passeggeri totali, ritardo medio (1 decimale) e percentuale di voli in ritardo (sulla colonna booleana si può usare `.mean()`: `True` vale 1).

In [ ]:
# TODO

#### 🧪 Esercizio – Passo 3: riepilogo per compagnia  ⭐⭐⭐
Crea la tabella `per_compagnia` con `agg`: numero di voli, passeggeri totali, ritardo medio (1 decimale), ordinata per passeggeri decrescenti. Salvala in `riepilogo_compagnie.csv` senza indice.

In [ ]:
# TODO

#### 🧪 Esercizio – Passo 4: quattro grafici  ⭐⭐⭐
Crea una figura 2×2 con: (1) barre dei passeggeri per compagnia, (2) istogramma dei ritardi, (3) linea dei passeggeri totali per giorno, (4) scatter distanza-ritardo. Ogni grafico con titolo ed etichette. Titolo generale `Voli di ottobre`.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 8))
# TODO
plt.tight_layout()
plt.show()

#### 🧪 Esercizio – Passo 5: salva e commenta  ⭐⭐
Salva la figura in `grafici/cruscotto_voli.png` (usa `fig.savefig` **prima** di `plt.show()`). Poi scrivi in una cella di testo tre osservazioni sui dati (quale compagnia trasporta più passeggeri, se i ritardi dipendono dalla distanza, quale giorno ha più traffico).

In [ ]:
from pathlib import Path
# TODO

## 8. 🔁 Riepilogo

### Cosa so fare oggi
- [ ] So installare pandas e matplotlib e importarli come `pd` e `plt`
- [ ] So creare `Series` e `DataFrame` e descriverli (`head`, `shape`, `info`, `describe`)
- [ ] So leggere e scrivere CSV e JSON con pandas
- [ ] So filtrare righe con una o più condizioni (`&`, `|`, `~`, `isin`, `between`) e ordinare
- [ ] So creare colonne calcolate (operazioni vettoriali, `np.where`, `apply` con lambda)
- [ ] So gestire valori mancanti, duplicati, testo e date sporchi
- [ ] So riassumere con `groupby`, `agg`, `value_counts`, `pivot_table` e unire con `merge`
- [ ] So disegnare grafici a linee, barre, scatter e istogrammi, con titoli, etichette e legenda
- [ ] So creare figure con più grafici e salvarle in un file

### Scheda di riferimento
| Voglio... | Comando |
|---|---|
| leggere un CSV | `pd.read_csv("file.csv", parse_dates=[...])` |
| filtrare | `df[(df["a"] > 1) & (df["b"] == "x")]` |
| scegliere colonne | `df[["a", "b"]]` |
| ordinare | `df.sort_values("a", ascending=False)` |
| nuova colonna | `df["c"] = df["a"] * 2` |
| valori mancanti | `df.isna().sum()`, `df.dropna()`, `df.fillna(0)` |
| riassumere per gruppo | `df.groupby("g").agg(tot=("a", "sum"))` |
| unire tabelle | `df1.merge(df2, left_on="x", right_on="y", how="left")` |
| grafico | `fig, ax = plt.subplots()` → `ax.plot/bar/scatter/hist` → `plt.show()` |

### Errori frequenti
| Messaggio | Causa e rimedio |
|---|---|
| `KeyError: 'nome'` | il nome della colonna è scritto male (maiuscole, spazi): controlla `df.columns` |
| `ValueError: The truth value of a Series is ambiguous` | hai usato `and` oppure `or` invece degli operatori `&` e `\|`, oppure mancano le parentesi attorno alle condizioni |
| il grafico non mostra nulla o è vuoto | manca `plt.show()`, oppure `savefig` è stato chiamato dopo `show` |
| i numeri risultano testo (`object`/`str`) | converti con `pd.to_numeric()`; controlla `df.dtypes` |
| `FileNotFoundError` | il file non è nella cartella corrente: controlla il percorso con `Path.cwd()` |

### Mini-quiz (a voce)
1. Qual è la differenza tra `df["a"]` e `df[["a"]]`?
2. Come si scrive il filtro "ITA e ritardo maggiore di 10"?
3. Cosa fa `groupby("compagnia")["passeggeri"].sum()`?
4. A cosa serve `how="left"` in un `merge`?
5. Perché un grafico a barre va ordinato prima di essere disegnato?
6. Cosa succede se si chiama `savefig` dopo `plt.show()`?

*Risposte:* 1) la prima è una `Series`, la seconda un `DataFrame` con una colonna · 2) `df[(df["compagnia"] == "ITA") & (df["ritardo_min"] > 10)]` · 3) somma i passeggeri per ogni compagnia · 4) mantiene tutte le righe della tabella di sinistra anche senza corrispondenza · 5) per leggere subito la classifica · 6) il file risulta vuoto, perché la figura è già stata chiusa

### Per continuare
`numpy` (calcoli su array), `seaborn` (grafici statistici più rapidi, costruito su matplotlib) e `scikit-learn` (primi modelli di machine learning) usano gli stessi concetti visti oggi.